# CS-781 Phase 5: Exp 2 — Layer-Wise Fine-Tuning (Model 3 GPT-NeoX)
Auto-generated for CS-781 Capstone Phase 5 (Legal Statute Identification).

In [ ]:
import os, json, glob, zipfile, shutil, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, get_linear_schedule_with_warmup
from torch.optim import AdamW
from huggingface_hub import hf_hub_download, HfApi
from sklearn.metrics import f1_score, precision_score, recall_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

HF_TOKEN = 'YOUR_HF_TOKEN'
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
TARGET_REPO = 'U-241111/cs781-submissions'

# 1. Download Phase 5 Dataset
os.makedirs('/kaggle/working/data_p5', exist_ok=True)
p5_files = [
    'Phase_5_LSI/Train/train.json',
    'Phase_5_LSI/Train/dev.json',
    'Phase_5_LSI/Test/test.json'
]
for f in p5_files:
    fname = os.path.basename(f)
    dest = f'/kaggle/working/data_p5/{fname}'
    if not os.path.exists(dest):
        print(f'Downloading {fname}...')
        loc = hf_hub_download(repo_id=HF_DATASET_REPO, filename=f, repo_type='dataset', token=HF_TOKEN)
        shutil.copyfile(loc, dest)
print('Downloaded all Phase 5 files successfully!')


In [ ]:
# 2. Locate Checkpoint and Initialize Tokenizer
model_ckpt_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'config.json' in files:
        model_ckpt_dir = root
        break
print('Model Checkpoint Path:', model_ckpt_dir, os.listdir(model_ckpt_dir))

tok = AutoTokenizer.from_pretrained(model_ckpt_dir)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token if tok.eos_token is not None else '<pad>'
tok.padding_side = 'left' # Crucial for causal sequence classification pooling

# Load datasets
with open('/kaggle/working/data_p5/train.json') as f:
    train_data = json.load(f)
with open('/kaggle/working/data_p5/dev.json') as f:
    dev_data = json.load(f)
with open('/kaggle/working/data_p5/test.json') as f:
    test_data = json.load(f)

print(f'Train: {len(train_data)} | Dev: {len(dev_data)} | Test: {len(test_data)}')

# Calculate positive weights for multi-label class imbalance
NUM_LABELS = 100
counts = np.zeros(NUM_LABELS, dtype=np.int64)
for d in train_data:
    for l in d['labels']:
        if 0 <= l < NUM_LABELS:
            counts[l] += 1
N = len(train_data)
raw_pw = np.sqrt((N - counts) / np.maximum(counts, 1))
clamped_pw = np.clip(raw_pw, 1.5, 12.0)
pos_weight = torch.tensor(clamped_pw, dtype=torch.float32).to(device)
print(f'Calculated pos_weight tensor: min={clamped_pw.min():.2f}, max={clamped_pw.max():.2f}, mean={clamped_pw.mean():.2f}')


In [ ]:
# 3. Multi-Window Spliced Dataset for Legal Statute Identification
class LSIDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=1024, is_test=False, num_labels=100):
        self.data = data
        self.tok = tokenizer
        self.max_length = max_length
        self.is_test = is_test
        self.num_labels = num_labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['text']
        tokens = self.tok.encode(text, add_special_tokens=False)
        
        if len(tokens) <= self.max_length:
            input_ids = tokens
        else:
            # Multi-window splice: First 384 tokens + Middle 384 tokens + Last 256 tokens = 1024
            first_chunk = tokens[:384]
            mid_idx = len(tokens) // 2
            mid_chunk = tokens[mid_idx - 192 : mid_idx + 192]
            last_chunk = tokens[-256:]
            input_ids = first_chunk + mid_chunk + last_chunk
            if len(input_ids) > self.max_length:
                input_ids = input_ids[:self.max_length]
        
        pad_len = self.max_length - len(input_ids)
        pad_id = self.tok.pad_token_id if self.tok.pad_token_id is not None else 0
        input_ids = [pad_id] * pad_len + input_ids
        attn_mask = [0] * pad_len + [1] * (self.max_length - pad_len)
        
        res = {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attn_mask, dtype=torch.long),
            'id': item['id']
        }
        if not self.is_test:
            target = torch.zeros(self.num_labels, dtype=torch.float32)
            for l in item['labels']:
                if 0 <= l < self.num_labels:
                    target[l] = 1.0
            res['target'] = target
        return res


In [ ]:
# 4. Initialize Model 3 (GPT-NeoX) with Partial / Layer-Wise Fine-Tuning
config = AutoConfig.from_pretrained(model_ckpt_dir, num_labels=100)
config.problem_type = 'multi_label_classification'
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt_dir, config=config)
model.config.pad_token_id = tok.pad_token_id

# Freeze lower layers, tune top 4 transformer blocks + score head
for name, p in model.named_parameters():
    p.requires_grad = False
    if any(f'layers.{i}.' in name for i in range(8, 12)) or 'score' in name or 'classifier' in name:
        p.requires_grad = True

model = model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Layer-Wise Model 3: Trainable {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

SUB_ZIP_NAME = 'submission_phase5_exp2_layerwise.zip'


In [ ]:
# Training Loop
train_loader = DataLoader(LSIDataset(train_data, tok, max_length=1024), batch_size=8, shuffle=True)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=4e-5, weight_decay=0.01)
epochs = 2
grad_accum = 4
total_steps = (len(train_loader) // grad_accum) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.08*total_steps), num_training_steps=total_steps)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f'Starting Layer-Wise training for {epochs} epochs...')
model.train()
for epoch in range(epochs):
    total_loss = 0
    t0 = time.time()
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        targets = batch['target'].to(device)
        logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
        loss = criterion(logits, targets) / grad_accum
        loss.backward()
        
        if (step + 1) % grad_accum == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        total_loss += loss.item() * grad_accum
    print(f'Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Time: {time.time()-t0:.1f}s')


In [ ]:
# 5. Evaluation, Per-Class Optimal Threshold Tuning, and Test Submission
val_loader = DataLoader(LSIDataset(dev_data, tok, max_length=1024), batch_size=16, shuffle=False)

model.eval()
val_preds_probs = []
val_targets = []
print('Running inference on Dev split...')
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        targets = batch['target'].to(device)
        logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
        probs = torch.sigmoid(logits).cpu().numpy()
        val_preds_probs.extend(probs)
        val_targets.extend(targets.cpu().numpy())

val_preds_probs = np.array(val_preds_probs)
val_targets = np.array(val_targets)

# Tune per-class optimal thresholds to directly maximize Macro-F1
optimal_thresholds = np.zeros(NUM_LABELS)
per_class_f1 = np.zeros(NUM_LABELS)

print('Optimizing per-class decision thresholds...')
for c in range(NUM_LABELS):
    best_f1 = 0.0
    best_th = 0.30
    for th in np.linspace(0.05, 0.60, 56):
        pred_c = (val_preds_probs[:, c] >= th).astype(int)
        f1_c = f1_score(val_targets[:, c], pred_c, zero_division=0)
        if f1_c > best_f1:
            best_f1 = f1_c
            best_th = th
    optimal_thresholds[c] = best_th
    per_class_f1[c] = best_f1

dev_macro_f1 = np.mean(per_class_f1)
print(f'>>> Calibrated Dev Macro-F1: {dev_macro_f1:.4f} <<<')
print(f'Mean optimal threshold: {np.mean(optimal_thresholds):.3f} (min: {np.min(optimal_thresholds):.3f}, max: {np.max(optimal_thresholds):.3f})')

# Inference on Test Set
test_loader = DataLoader(LSIDataset(test_data, tok, max_length=1024, is_test=True), batch_size=16, shuffle=False)
test_ids = []
test_probs = []

print('Running inference on Test split...')
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        logits = model(input_ids=input_ids, attention_mask=attn_mask).logits
        probs = torch.sigmoid(logits).cpu().numpy()
        test_probs.extend(probs)
        test_ids.extend(batch['id'])

test_probs = np.array(test_probs)

# Apply calibrated thresholds to generate official multi-label statute predictions
final_predictions = []
for i, did in enumerate(test_ids):
    selected = [int(c) for c in range(NUM_LABELS) if test_probs[i, c] >= optimal_thresholds[c]]
    if not selected:
        selected = [int(np.argmax(test_probs[i]))]
    final_predictions.append({
        'id': str(did),
        'pred': selected
    })

assert len(final_predictions) == len(test_data), f'Mismatch: {len(final_predictions)} vs {len(test_data)}'
print(f'Generated predictions for {len(final_predictions)} test cases.')

# Save prediction.json
with open('/kaggle/working/prediction.json', 'w', encoding='utf-8') as f:
    json.dump(final_predictions, f, indent=2)

# Save probabilities and thresholds for ensembling
np.save('/kaggle/working/test_probs.npy', test_probs)
np.save('/kaggle/working/val_preds_probs.npy', val_preds_probs)
np.save('/kaggle/working/optimal_thresholds.npy', optimal_thresholds)

# Create submission zip
with zipfile.ZipFile(f'/kaggle/working/{SUB_ZIP_NAME}', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('/kaggle/working/prediction.json', arcname='prediction.json')

print(f'Created {SUB_ZIP_NAME} successfully!')

# Auto-upload to Hugging Face repository
try:
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj=f'/kaggle/working/{SUB_ZIP_NAME}',
        path_in_repo=f'phase5/{SUB_ZIP_NAME}',
        repo_id=TARGET_REPO,
        repo_type='dataset'
    )
    api.upload_file(
        path_or_fileobj='/kaggle/working/test_probs.npy',
        path_in_repo=f"phase5/{SUB_ZIP_NAME.replace('.zip', '_test_probs.npy')}",
        repo_id=TARGET_REPO,
        repo_type='dataset'
    )
    api.upload_file(
        path_or_fileobj='/kaggle/working/optimal_thresholds.npy',
        path_in_repo=f"phase5/{SUB_ZIP_NAME.replace('.zip', '_thresholds.npy')}",
        repo_id=TARGET_REPO,
        repo_type='dataset'
    )
    print('Uploaded all Phase 5 submission artifacts to HF!')
except Exception as e:
    print('HF upload exception:', e)
